# E-commerce Conversion & Revenue Funnel Analytics

## Data Understanding & Data Quality Assessment

### Business Objective

Analyze the e-commerce customer journey from product view
to add-to-cart to purchase, identify funnel bottlenecks,
and uncover opportunities to improve conversion and revenue.

### Funnel

Product View → Add to Cart → Purchase

### Dataset

eCommerce Behavior Data from Multi-Category Store
Period: October 2019

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
DATA_PATH = Path("../data/raw/2019-Oct.csv")

print(DATA_PATH.exists())
print(DATA_PATH)

True
..\data\raw\2019-Oct.csv


## Sample Data Inspection

A 10,000-row sample is inspected first to understand the dataset structure, column names, data types, event types, and initial missing-value patterns before performing full-dataset analysis.

In [4]:
sample_df = pd.read_csv(
    DATA_PATH,
    nrows=10000
)

print(sample_df.shape)
print(sample_df.columns.tolist())
sample_df.head()

(10000, 9)
['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00 UTC,view,44600062,2103807459595387724,NaN,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00 UTC,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01 UTC,view,17200506,2053013559792632471,furniture.living_room.sofa,NaN,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
3,2019-10-01 00:00:01 UTC,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
4,2019-10-01 00:00:04 UTC,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d


In [5]:
sample_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   event_time     10000 non-null  object 
 1   event_type     10000 non-null  object 
 2   product_id     10000 non-null  int64  
 3   category_id    10000 non-null  int64  
 4   category_code  6723 non-null   object 
 5   brand          8558 non-null   object 
 6   price          10000 non-null  float64
 7   user_id        10000 non-null  int64  
 8   user_session   10000 non-null  object 
dtypes: float64(1), int64(3), object(5)
memory usage: 703.3+ KB


In [6]:
sample_df["event_type"].value_counts(dropna=False)

event_type
view        9785
purchase     118
cart          97
Name: count, dtype: int64

In [7]:
sample_df.isnull().sum()

event_time          0
event_type          0
product_id          0
category_id         0
category_code    3277
brand            1442
price               0
user_id             0
user_session        0
dtype: int64

# Full Dataset Inspection

In [8]:
#Full row count
total_rows = 0

for chunk in pd.read_csv(
    DATA_PATH,
    chunksize=500_000
):
    total_rows += len(chunk)

print(f"Total rows: {total_rows:,}")

Total rows: 42,448,764


In [9]:
#Full event distribution
event_counts = {}

for chunk in pd.read_csv(
    DATA_PATH,
    chunksize=500_000,
    usecols=["event_type"]
):
    counts = chunk["event_type"].value_counts()

    for event, count in counts.items():
        event_counts[event] = (
            event_counts.get(event, 0) + count
        )

event_counts = pd.Series(event_counts).sort_values(
    ascending=False
)

event_counts

view        40779399
cart          926516
purchase      742849
dtype: int64

The dataset contains three event types with recorded events:

view → cart → purchase

In [10]:
#Full unique users
unique_users = set()

for chunk in pd.read_csv(
    DATA_PATH,
    chunksize=500_000,
    usecols=["user_id"]
):
    unique_users.update(
        chunk["user_id"].dropna().unique()
    )

print(f"Unique users: {len(unique_users):,}")

Unique users: 3,022,290


In [11]:
#Full unique sessions
unique_sessions = set()

for chunk in pd.read_csv(
    DATA_PATH,
    chunksize=500_000,
    usecols=["user_session"]
):
    unique_sessions.update(
        chunk["user_session"].dropna().unique()
    )

print(f"Unique sessions: {len(unique_sessions):,}")

Unique sessions: 9,244,421


In [12]:
#Full date range
min_date = None
max_date = None

for chunk in pd.read_csv(
    DATA_PATH,
    chunksize=500_000,
    usecols=["event_time"]
):
    dates = pd.to_datetime(
        chunk["event_time"],
        errors="coerce"
    )

    chunk_min = dates.min()
    chunk_max = dates.max()

    if min_date is None or chunk_min < min_date:
        min_date = chunk_min

    if max_date is None or chunk_max > max_date:
        max_date = chunk_max

print("Minimum:", min_date)
print("Maximum:", max_date)

Minimum: 2019-10-01 00:00:00+00:00
Maximum: 2019-10-31 23:59:59+00:00


In [13]:
#Full product/category counts
unique_products = set()
unique_categories = set()

for chunk in pd.read_csv(
    DATA_PATH,
    chunksize=500_000,
    usecols=["product_id", "category_id"]
):
    unique_products.update(
        chunk["product_id"].dropna().unique()
    )

    unique_categories.update(
        chunk["category_id"].dropna().unique()
    )

print(f"Unique products: {len(unique_products):,}")
print(f"Unique categories: {len(unique_categories):,}")

Unique products: 166,794
Unique categories: 624


In [14]:
#Full missing-value assessment
missing_counts = {}

total_rows = 0

for chunk in pd.read_csv(
    DATA_PATH,
    chunksize=500_000
):
    total_rows += len(chunk)

    counts = chunk.isna().sum()

    for column, count in counts.items():
        missing_counts[column] = (
            missing_counts.get(column, 0) + count
        )

missing_df = pd.DataFrame({
    "missing_count": missing_counts
})

missing_df["missing_pct"] = (
    missing_df["missing_count"] / total_rows * 100
)

missing_df.sort_values(
    "missing_pct",
    ascending=False
)

,missing_count,missing_pct
category_code,13515609,31.839818
brand,6117080,14.410502
user_session,2,0.000005
event_time,0,0.000000
event_type,0,0.000000
product_id,0,0.000000
category_id,0,0.000000
price,0,0.000000
user_id,0,0.000000


In [15]:
#Full duplicate assessment
duplicate_rows = 0

for chunk in pd.read_csv(
    DATA_PATH,
    chunksize=500_000
):
    duplicate_rows += chunk.duplicated().sum()

print(f"Duplicate rows found within chunks: {duplicate_rows:,}")

Duplicate rows found within chunks: 30,219


In [17]:
#Minimum and maximum price
price_min = float("inf")
price_max = float("-inf")

for chunk in pd.read_csv(
    DATA_PATH,
    chunksize=500_000,
    usecols=["price"]
):
    price_min = min(price_min, chunk["price"].min())
    price_max = max(price_max, chunk["price"].max())

print("Minimum price:", price_min)
print("Maximum price:", price_max)

Minimum price: 0.0
Maximum price: 2574.07


In [18]:
#How many zero-price records?
zero_price_count = 0

for chunk in pd.read_csv(
    DATA_PATH,
    chunksize=500_000,
    usecols=["price"]
):
    zero_price_count += (chunk["price"] == 0).sum()

print(f"Zero-price records: {zero_price_count:,}")
print(f"Percentage: {zero_price_count / total_rows * 100:.4f}%")

Zero-price records: 68,673
Percentage: 0.1618%


In [19]:
#Zero-price records by event type
zero_price_by_event = {}

for chunk in pd.read_csv(
    DATA_PATH,
    chunksize=500_000,
    usecols=["event_type", "price"]
):
    zero_rows = chunk[chunk["price"] == 0]

    counts = zero_rows["event_type"].value_counts()

    for event, count in counts.items():
        zero_price_by_event[event] = (
            zero_price_by_event.get(event, 0) + count
        )

print(pd.Series(zero_price_by_event).sort_values(ascending=False))

view    68523
cart      150
dtype: int64


In [20]:
#Missing category_code and brand by event type
missing_by_event = {}

for chunk in pd.read_csv(
    DATA_PATH,
    chunksize=500_000,
    usecols=["event_type", "category_code", "brand"]
):
    grouped = chunk.groupby("event_type")[["category_code", "brand"]].apply(
        lambda x: x.isna().sum()
    )

    for event, row in grouped.iterrows():
        if event not in missing_by_event:
            missing_by_event[event] = {
                "category_code": 0,
                "brand": 0
            }

        missing_by_event[event]["category_code"] += row["category_code"]
        missing_by_event[event]["brand"] += row["brand"]

missing_by_event = pd.DataFrame(missing_by_event).T
print(missing_by_event)

          category_code    brand
cart             105726    18806
purchase         173425    58305
view           13236458  6039969


# Data Understanding & Data Quality Summary

## Dataset

- Dataset: eCommerce Behavior Data from Multi-Category Store
- Analysis period: October 2019
- File: `2019-Oct.csv`
- Total events: 42,448,764
- Number of columns: 9

## Dataset Scale

| Metric | Result |
|---|---:|
| Total events | 42,448,764 |
| Unique users | 3,022,290 |
| Unique sessions | 9,244,421 |
| Unique products | 166,794 |
| Unique categories | 624 |
| Start date | 2019-10-01 00:00:00 UTC |
| End date | 2019-10-31 23:59:59 UTC |

## Event Distribution

| Event Type | Event Count |
|---|---:|
| View | 40,779,399 |
| Cart | 926,516 |
| Purchase | 742,849 |
| Remove from cart | 0 |

The dataset contains three observed event types:

`view → cart → purchase`

Event counts represent the number of recorded events, not unique users. Therefore, event counts will not be used directly as funnel conversion rates. User-level funnel logic will be developed.

## Missing Values

| Column | Missing Count | Missing % |
|---|---:|---:|
| category_code | 13,515,609 | 31.84% |
| brand | 6,117,080 | 14.41% |
| user_session | 2 | 0.000005% |
| event_time | 0 | 0% |
| event_type | 0 | 0% |
| product_id | 0 | 0% |
| category_id | 0 | 0% |
| price | 0 | 0% |
| user_id | 0 | 0% |

### Missing Value Assessment

`category_code` and `brand` contain a significant number of missing values. However, these are descriptive dimensions rather than core identifiers required for the primary funnel.

Rows will not be deleted solely because `category_code` or `brand` is missing.

Missing categorical values will be handled explicitly so that potentially valuable events, including purchases, are not unnecessarily removed.

## Zero-Price Records

- Zero-price records: 68,673
- Percentage of dataset: 0.1618%

### Zero-Price Distribution by Event Type

| Event Type | Zero-Price Records |
|---|---:|
| View | 68,523 |
| Cart | 150 |
| Purchase | 0 |

Zero-price records represent a very small proportion of the dataset.

Because no purchase events have a zero price, these records will not be treated as automatic errors or deleted at this stage.

Further validation will be performed before price-based analysis.

## Duplicate Records

At least 30,219 exact duplicate rows were identified within individual 500,000-row chunks.

This does not necessarily represent the complete number of duplicates in the dataset because duplicates spanning chunk boundaries were not detected by the initial check.

A stronger duplicate validation will be performed during cleaning.

## Price Range

- Minimum price: 0.00
- Maximum price: 2,574.07

The price distribution will be analyzed further before creating price segments or bands.

## Initial Data Quality Assessment

### Strengths

- Core event fields contain no missing values.
- `user_id` contains no missing values.
- `product_id` contains no missing values.
- `category_id` contains no missing values.
- `event_time` contains no missing values.
- `price` contains no missing values.
- The dataset covers the complete October 2019 period.
- The required funnel events are available.

### Data Quality Issues

1. `category_code` has 31.84% missing values.
2. `brand` has 14.41% missing values.
3. Two records have missing `user_session`.
4. At least 30,219 duplicate rows were detected within chunks.
5. 68,673 records have a price of zero.
6. The dataset is very large, with 42.45 million events, so chunk-based processing is required.

## Key Analytical Decisions

Based on the investigation:

1. Do not calculate funnel conversion rates from raw event counts.
2. Build the primary funnel at the user level.
3. Do not delete rows solely because `brand` or `category_code` is missing.
4. Investigate zero-price records before price-based analysis.
5. Perform stronger duplicate validation during cleaning.
6. Use chunk-based processing because the raw dataset is too large for a simple full-file pandas load.

## Conclusion

The October 2019 dataset contains 42.45 million e-commerce events generated by approximately 3.02 million unique users across 9.24 million sessions.

The dataset is suitable for the planned E-commerce Conversion & Revenue Funnel Analytics project. Core funnel fields are complete, while missing values are concentrated mainly in descriptive attributes such as category_code and brand.

The main analytical challenge is the scale of the dataset, requiring memory-efficient processing. Next step will focus on cleaning, validation, transformation, and construction of the user-level funnel dataset.